In [7]:
import os
import random

# dataset
from lerobot.datasets.transforms import ImageTransformsConfig
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

# policy
from lerobot.policies.act.configuration_act import ACTConfig

# lerobot
from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.configs.default import WandBConfig
from lerobot.utils.utils import init_logging
from lerobot.configs.types import PolicyFeature, FeatureType

# torch
from torch.cuda import is_available

# wandb
import wandb

# utils
from dotenv import load_dotenv
from pprint import pprint
import sys

# my code
from src.paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR
from src.train_extended import train_extended
from src.yolo.yolo_policy_preprocessor import FilterEnvObsProcessorStep, RemoveFeatureProcessorStep

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# magic autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
device = "cuda" if is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [9]:
PUSH_TO_HUB       = True
WANDB             = True
REPO_NAME         = 'so101_pick_pen'
EXPERIMENT_NAME   = 'v4'
RESUME_CHECKPOINT = None
DATASET_PATH      = DATASETS_DIR / REPO_NAME
DATASET_ID        = f"{HF_NAME}/{REPO_NAME}"
POLICY_ID         = f"{HF_NAME}/act-{REPO_NAME}-{EXPERIMENT_NAME}"

# define episodes to train on -  if all, select None
# episodes          = random.sample(range(0, 50), 25)
episodes = None

# features
POLICY_REMOVE_FEATURES   = True

In [10]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")
if WANDB:
    wandb.login(key = os.getenv('WANDB_API_KEY'))

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `pc` has been saved to /home/jonathan/.cache/huggingface/stored_tokens
Your token has been saved to /home/jonathan/.cache/huggingface/token
Login successful.
The current active token is: `pc`
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/jonathan/.netrc


In [11]:
ds_meta = LeRobotDatasetMetadata(DATASET_ID, root = DATASET_PATH)
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Total Frames: {ds_meta.total_frames}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"\nRobot type: {ds_meta.robot_type}")
print(f"\nkeys to access images from cameras: {ds_meta.camera_keys}")
print("\nFeatures:")
pprint(ds_meta.features.keys(), width = 40 )

Total number of episodes: 76
Average number of frames per episode: 683.000
Total Frames: 51908
Frames per second used during data collection: 30

Robot type: so101_follower

keys to access images from cameras: ['observation.images.wrist_cam', 'observation.images.top_cam']

Features:
dict_keys(['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'])


In [12]:
# select image transforms
transforms_cfg = ImageTransformsConfig(
    enable             = False, # note about augmentations
    # max_num_transforms = 3,
    # random_order       = True,
)

# build dataset
dataset_cfg = DatasetConfig(
    repo_id            = DATASET_ID,
    root               = DATASET_PATH.__str__(),   # dataset location
    image_transforms   = transforms_cfg,           # configure image transformations
    use_imagenet_stats = True,                     # normalize image using imagenet weights
    video_backend      = 'torchcodec',
    streaming          = False,
    episodes           = episodes
)

### Policy Config
Manually configure policy features - this overrides defaults from the dataset:

In [13]:
input_features = {
    'observation.state': PolicyFeature(
        type  = FeatureType.STATE,
        shape = (6,) #  !!! NOTICE TODO CAUTION - using 12 for current sensing !!
    ),
    'observation.images.wrist_cam': PolicyFeature(
        type  = FeatureType.VISUAL,
        shape = (3,480, 640)
    ),
    'observation.images.top_cam': PolicyFeature(
        type  = FeatureType.VISUAL,
        shape = (3, 480, 640)
    ),
    # 'observation.environment_state': PolicyFeature(
    #     type  = FeatureType.ENV,
    #     shape = (4,)
    # )
}
output_features = {
    'action': PolicyFeature(
        type  = FeatureType.ACTION,
        shape = (6,)
    )
}

Additional processors: if needed, specify here:

In [14]:
if POLICY_REMOVE_FEATURES:
    policy_processors = [
        # RemoveFeatureProcessorStep(['observation.images.top_cam', 'observation.images.wrist_cam'])
        FilterEnvObsProcessorStep(feature_name = 'observation.state', remove_indices = list(range(6, 12)))
    ]
else:
    policy_processors = None

In [15]:
policy_config = ACTConfig(
    n_obs_steps                 = 1,                                  # policy sample freq - should be 1
    chunk_size                  = 100,                                # chunk size outut from the moodel
    n_action_steps              = 100,                                # use only 15 at inference (0.5 sec)
    input_features              = input_features,
    output_features             = output_features,
    device                      = device,
    n_encoder_layers            = 4, #default 4
    n_decoder_layers            = 1, #default 1
    use_amp                     = False,                              # note about amp
    push_to_hub                 = PUSH_TO_HUB,
    repo_id                     = POLICY_ID,
    vision_backbone             = 'resnet18',
    pretrained_backbone_weights = 'ResNet18_Weights.IMAGENET1K_V1',
    dim_model                   = 512,
    use_vae                     = True,                               # should you use a encoder to learn the style variable
    latent_dim                  = 32,                                 # of the latent Z encoder
    n_vae_encoder_layers        = 4,                                  # of the latent Z encoder
    temporal_ensemble_coeff     = None,                               # temporal averaging coefficient, nominal is 0.01
    kl_weight                   = 10,                                 # defualt is 10
    optimizer_lr                = 1e-5                                # default is 1e-5
    )

WandB logging:

In [16]:
wandb_config = WandBConfig(
    enable           = WANDB,
    disable_artifact = True,
    project          = 'ACT-Real-Sim-Real',
    entity           = 'jonathm126-personal',
    mode             = 'online',
    run_id           = f'act-{REPO_NAME}-{EXPERIMENT_NAME}-train'  # careful - this is a unique ids
)
print(wandb_config.run_id)


act-so101_pick_pen-v4-train


Integration:

In [17]:
train_cfg = TrainPipelineConfig(
    dataset                    = dataset_cfg,
    # env                        = env_config,
    policy                     = policy_config,                                                     # policy config
    output_dir                 = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME,
    job_name                   = POLICY_ID + '-train',                                              # name
    num_workers                = 4,
    batch_size                 = 12,                                                                # check for stabillity
    steps                      = 100000,                                                             # check for number of epochs - we want 15-20 epochs
    eval_freq                  = 20000,
    log_freq                   = 200,
    save_checkpoint            = True,
    save_freq                  = 25000,
    use_policy_training_preset = True,                                                              # for scheduler and optimizer
    wandb                      = wandb_config,
    resume                     = RESUME_CHECKPOINT is not None
)

Train:

In [18]:
if RESUME_CHECKPOINT is None:
    init_logging()
    train_extended(train_cfg, extra_pipeline = policy_processors)
else:
    assert RESUME_CHECKPOINT is not None
    pth = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / RESUME_CHECKPOINT / 'pretrained_model' / 'train_config.json'
    assert pth.exists()
    sys.argv = [
        "train",
        f"--config_path={str(pth)}",
        "--resume=true",
    ]
    init_logging()
    train_extended(extra_pipeline = policy_processors)

INFO 2026-02-05 21:49:11 extended.py:148 {'batch_size': 12,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         'type': 'ColorJitter',
                                                         'weight': 1.0},
                                          'contrast': {'kwargs': {'contrast': [0.8,
                                                                               1.2]},
                                                       'type': 'ColorJitter',
                                                       'weight': 1.0},
                                          'hue': {'kwargs': {'hue': [-0.05,
                

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.
INFO 2026-02-05 21:49:12 db_utils.py:103 Track this run --> https://wandb.ai/jonathm126-personal/ACT-Real-Sim-Real/runs/act-so101_pick_pen-v4-train
INFO 2026-02-05 21:49:12 extended.py:164 Creating dataset


Logs will be synced with wandb.


INFO 2026-02-05 21:49:13 extended.py:175 Creating policy
INFO 2026-02-05 21:49:14 extended.py:212 Creating optimizer and scheduler
INFO 2026-02-05 21:49:14 extended.py:224 Output dir: /home/jonathan/Github/IDC_Project-Sim_Real_Sim/policies/act/so101_pick_pen/v4
INFO 2026-02-05 21:49:14 extended.py:227 cfg.steps=100000 (100K)
INFO 2026-02-05 21:49:14 extended.py:228 dataset.num_frames=51908 (52K)
INFO 2026-02-05 21:49:14 extended.py:229 dataset.num_episodes=76
INFO 2026-02-05 21:49:14 extended.py:230 num_learnable_params=51597190 (52M)
INFO 2026-02-05 21:49:14 extended.py:231 num_total_params=51597190 (52M)
INFO 2026-02-05 21:49:14 extended.py:273 Start offline training on a fixed dataset
INFO 2026-02-05 21:50:27 extended.py:300 step:200 smpl:2K ep:4 epch:0.05 loss:6.797 grdn:135.170 lr:1.0e-05 updt_s:0.355 data_s:0.009
INFO 2026-02-05 21:51:36 extended.py:300 step:400 smpl:5K ep:7 epch:0.09 loss:2.808 grdn:73.986 lr:1.0e-05 updt_s:0.344 data_s:0.005
INFO 2026-02-05 21:52:47 extended.py